# 상황

로지스틱 회귀는 해석에 용이하고 우리 목적엔 맞지 않는 모델이라고 판단했다~~~~(간단하게)

랜덤포레스트는 초기 트리모델로 우리 목적엔 이러한 이유로 맞지 않는다고 판단했다~~~~(간단하게)

xgboost랑 lightGBM은 이러한 이유로 우리 목적에 맞는 알고리즘이고, 성능과 여러 비교 분석을 통해 lightGBM으로 결정했다. 

# 최종 선정 순서

1. 동일 Fold·조합3·동일 가중치 계산 조건에서 두 모델 평가
2. Fold별 및 평균 PR-AUC 비교
3. PR-AUC 표준편차와 최저 Fold 성능 비교
4. 시간순 Fold Validation 예측확률 통합
5. 각 모델의 F1 최대 임계값과 성능 계산
6. 동일 목표 Recall에서 Precision·FP·FN 비교
7. Train–Validation 격차 확인
8. Fold별 임계값과 임계값 주변 민감도 확인
9. 필요하면 확률 보정 상태 비교
10. 성능이 실질적으로 같으면 학습·추론 효율로 최종 선택

xgboost 나겸님 코드가 완성되지 않은 관계로, 

lightGBM 에서 했던 학습 조건을 xgboost도 통일해서 xgboost 학습(조합 3만)

In [2]:
%pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB 325.1 kB/s eta 0:05:13
   ---------------------------------------- 0.1/101.7 MB 819.2 kB/s eta 0:02:04
   ---------------------------------------- 0.6/101.7 MB 3.3 MB/s eta 0:00:31
   ---------------------------------------- 1.3/101.7 MB 5.8 MB/s eta 0:00:18
    --------------------------------------- 2.3/101.7 MB 8.6 MB/s eta 0:00:12
   - -------------------------------------- 3.4/101.7 MB 10.7 MB/s eta 0:00:10
   - -------------------------------------- 4.9/101.7 MB 13.5 MB/s eta 0:00:08
   -- ------------------------------------- 6.4/101.7 MB 15.7 MB/s eta 0:00:07
   --- ------------------------------------ 8.3/101.7 MB 18.8 MB/s eta 0:00:05
   ---- ----------------------------------- 10.2/101.7 MB 21.1 MB/s eta 0:00:


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys

print("현재 커널:", sys.executable)

try:
    import xgboost
    print("XGBoost 설치됨:", xgboost.__version__)
except ModuleNotFoundError:
    print("XGBoost 설치 안 됨")

현재 커널: c:\Users\splen\OneDrive\Desktop\BDAI_\BDAI\.venv\Scripts\python.exe
XGBoost 설치됨: 3.2.0


In [3]:
# ============================================================
# XGBoost 실행 전 준비
# 1. 데이터 불러오기
# 2. 시간순 정렬
# 3. LightGBM과 동일한 행 기준 3-Fold 생성
# ============================================================

import numpy as np
import pandas as pd
import xgboost as xgb

from xgboost import XGBClassifier


# ============================================================
# 1. 기본 설정
# ============================================================

DATA_PATH = (
    r"C:\Users\splen\OneDrive\Desktop\BDAI_"
    r"\BOOSTMAP\Fraud-FDS-Project\data"
    r"\fraud_full_features.csv"
)

TARGET = "is_fraud"
TIME_COL = "trans_date_trans_time"
RANDOM_STATE = 42
COMMON_THRESHOLD = 0.90


# ============================================================
# 2. 데이터 불러오기
# ============================================================

df = pd.read_csv(DATA_PATH)

print("✅ 데이터 불러오기 완료")
print("전체 데이터 크기:", df.shape)


# ============================================================
# 3. 필수 변수 확인
# ============================================================

COMBINATION3_FEATURES = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",
    "high_speed"
]

required_columns = (
    [TIME_COL, TARGET]
    + COMBINATION3_FEATURES
)

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "다음 필수 변수가 데이터에 없습니다: "
        + ", ".join(missing_columns)
    )

print("✅ 조합3 필수 변수 확인 완료")


# ============================================================
# 4. 시간 변수 변환 및 정렬
# ============================================================

df[TIME_COL] = pd.to_datetime(
    df[TIME_COL],
    errors="raise"
)

df = (
    df.sort_values(TIME_COL)
      .reset_index(drop=True)
)

print("✅ 시간순 정렬 완료")
print(
    "전체 기간:",
    df[TIME_COL].min(),
    "~",
    df[TIME_COL].max()
)

print("전체 정상거래 수:", int((df[TARGET] == 0).sum()))
print("전체 이상거래 수:", int((df[TARGET] == 1).sum()))
print(f"전체 이상거래율: {df[TARGET].mean() * 100:.6f}%")


# ============================================================
# 5. 전체 데이터를 행 수 기준 6개 구간으로 분할
# ============================================================
#
# Fold 1: 앞 3구간 학습 → 4번째 구간 검증
# Fold 2: 앞 4구간 학습 → 5번째 구간 검증
# Fold 3: 앞 5구간 학습 → 6번째 구간 검증
#
# ============================================================

number_of_rows = len(df)

boundaries = np.linspace(
    0,
    number_of_rows,
    7,
    dtype=int
)

print("\n6개 구간 경계:", boundaries)


# ============================================================
# 6. LightGBM과 동일한 Expanding Window 3-Fold 생성
# ============================================================

time_folds = []

for fold_number in range(1, 4):

    train_end = boundaries[fold_number + 2]
    validation_start = train_end
    validation_end = boundaries[fold_number + 3]

    train_idx_fold = np.arange(
        0,
        train_end
    )

    validation_idx_fold = np.arange(
        validation_start,
        validation_end
    )

    time_folds.append({
        "fold": fold_number,
        "train_idx": train_idx_fold,
        "val_idx": validation_idx_fold
    })


# ============================================================
# 7. Fold별 기간과 이상거래 분포 확인
# ============================================================

fold_check_results = []

for fold_info in time_folds:

    fold_number = fold_info["fold"]

    train_part = df.iloc[
        fold_info["train_idx"]
    ]

    validation_part = df.iloc[
        fold_info["val_idx"]
    ]

    fold_scale_pos_weight = (
        (train_part[TARGET] == 0).sum()
        /
        (train_part[TARGET] == 1).sum()
    )

    fold_check_results.append({
        "fold": fold_number,

        "train_start":
            train_part[TIME_COL].min(),

        "train_end":
            train_part[TIME_COL].max(),

        "validation_start":
            validation_part[TIME_COL].min(),

        "validation_end":
            validation_part[TIME_COL].max(),

        "train_count":
            len(train_part),

        "validation_count":
            len(validation_part),

        "train_positive":
            int(train_part[TARGET].sum()),

        "validation_positive":
            int(validation_part[TARGET].sum()),

        "validation_fraud_rate":
            validation_part[TARGET].mean(),

        "scale_pos_weight":
            fold_scale_pos_weight,

        "time_order_correct":
            (
                train_part[TIME_COL].max()
                <= validation_part[TIME_COL].min()
            )
    })


fold_check_df = pd.DataFrame(
    fold_check_results
)

print("\n")
print("=" * 80)
print("행 기준 Expanding Window 3-Fold 확인")
print("=" * 80)

display(fold_check_df)


# ============================================================
# 8. 최종 확인
# ============================================================

if len(time_folds) != 3:
    raise ValueError("Fold가 정확히 3개 생성되지 않았습니다.")

if not fold_check_df["time_order_correct"].all():
    raise ValueError(
        "Train과 Validation의 시간 순서에 문제가 있습니다."
    )

print("\n✅ df 준비 완료")
print("✅ time_folds 생성 완료")
print("✅ LightGBM과 동일한 3-Fold 구간 확인 완료")
print("XGBoost 버전:", xgb.__version__)

✅ 데이터 불러오기 완료
전체 데이터 크기: (1296675, 31)
✅ 조합3 필수 변수 확인 완료
✅ 시간순 정렬 완료
전체 기간: 2019-01-01 00:00:18 ~ 2020-06-21 12:13:37
전체 정상거래 수: 1289169
전체 이상거래 수: 7506
전체 이상거래율: 0.578865%

6개 구간 경계: [      0  216112  432225  648337  864450 1080562 1296675]


행 기준 Expanding Window 3-Fold 확인


,fold,train_start,train_end,validation_start,validation_end,train_count,validation_count,train_positive,validation_positive,validation_fraud_rate,scale_pos_weight,time_order_correct
0,1,2019-01-01 00:00:18,2019-10-03 07:35:11,2019-10-03 07:35:47,2019-12-18 17:06:47,648337,216113,3827,1091,0.005048,168.411288,True
1,2,2019-01-01 00:00:18,2019-12-18 17:06:47,2019-12-18 17:07:05,2020-03-24 15:14:25,864450,216112,4918,1363,0.006307,174.772672,True
2,3,2019-01-01 00:00:18,2020-03-24 15:14:25,2020-03-24 15:14:30,2020-06-21 12:13:37,1080562,216113,6281,1225,0.005668,171.036618,True



✅ df 준비 완료
✅ time_folds 생성 완료
✅ LightGBM과 동일한 3-Fold 구간 확인 완료
XGBoost 버전: 3.2.0


In [4]:
# ============================================================
# XGBoost 조합3
# LightGBM과 동일한 행 기준 Expanding Window 3-Fold
# ============================================================

import gc
import time
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from xgboost import XGBClassifier


# ============================================================
# 1. 공통 설정
# ============================================================

TARGET = "is_fraud"
TIME_COL = "trans_date_trans_time"
COMMON_THRESHOLD = 0.90
RANDOM_STATE = 42

# XGBoost 담당자가 Model 44(조합3)에서 확정한 최적 설정
XGB_PARAMS = {
    "n_estimators": 2000,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "gamma": 0.1,
    "reg_alpha": 0.1,
    "reg_lambda": 3,
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "early_stopping_rounds": 100,
    "random_state": RANDOM_STATE,
    "n_jobs": -1
}


# ============================================================
# 2. 조합3 변수
# ============================================================

XGB_COMBINATION3_FEATURES = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",
    "high_speed"
]

CATEGORICAL_FEATURES = ["category"]

NUMERIC_FEATURES = [
    feature
    for feature in XGB_COMBINATION3_FEATURES
    if feature not in CATEGORICAL_FEATURES
]


# ============================================================
# 3. 사전 확인
# ============================================================

if "df" not in globals():
    raise NameError("df가 없습니다. 데이터 불러오기 셀부터 실행해주세요.")

if "time_folds" not in globals():
    raise NameError(
        "time_folds가 없습니다. "
        "LightGBM의 행 기준 3-Fold 생성 셀을 먼저 실행해주세요."
    )

if len(time_folds) != 3:
    raise ValueError(
        f"time_folds에는 3개 Fold가 있어야 합니다. "
        f"현재 Fold 수: {len(time_folds)}"
    )

missing_columns = [
    column
    for column in XGB_COMBINATION3_FEATURES + [TARGET, TIME_COL]
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "다음 변수가 데이터에 없습니다: "
        + ", ".join(missing_columns)
    )

# 시간순 정렬 재확인
df[TIME_COL] = pd.to_datetime(df[TIME_COL])

df = (
    df.sort_values(TIME_COL)
      .reset_index(drop=True)
)

print("✅ XGBoost 조합3 학습 준비 완료")
print("Fold 수:", len(time_folds))
print("원본 변수 수:", len(XGB_COMBINATION3_FEATURES))


# ============================================================
# 4. 결과 저장 공간
# ============================================================

xgb_fold_results = []
xgb_oof_predictions = []
xgb_fold_models = []

total_start_time = time.perf_counter()


# ============================================================
# 5. 시간순 3-Fold 학습
# ============================================================

for fold_info in time_folds:

    fold_number = fold_info["fold"]
    fold_start_time = time.perf_counter()

    train_idx = fold_info["train_idx"]
    val_idx = fold_info["val_idx"]

    train_fold = df.iloc[train_idx].copy()
    val_fold = df.iloc[val_idx].copy()

    y_train_fold = (
        train_fold[TARGET]
        .astype(np.int8)
        .reset_index(drop=True)
    )

    y_val_fold = (
        val_fold[TARGET]
        .astype(np.int8)
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Fold Train 기준으로 가중치 계산
    # --------------------------------------------------------

    negative_count = int((y_train_fold == 0).sum())
    positive_count = int((y_train_fold == 1).sum())

    if positive_count == 0:
        raise ValueError(
            f"Fold {fold_number} Train에 이상거래가 없습니다."
        )

    fold_scale_pos_weight = (
        negative_count / positive_count
    )

    # --------------------------------------------------------
    # 범주형 변수 One-Hot Encoding
    # 반드시 해당 Fold Train에만 fit
    # --------------------------------------------------------

    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True,
        dtype=np.float32
    )

    X_train_category = encoder.fit_transform(
        train_fold[CATEGORICAL_FEATURES]
        .astype("string")
        .fillna("missing")
    )

    X_val_category = encoder.transform(
        val_fold[CATEGORICAL_FEATURES]
        .astype("string")
        .fillna("missing")
    )

    # --------------------------------------------------------
    # 수치형 변수
    # --------------------------------------------------------

    X_train_numeric = csr_matrix(
        train_fold[NUMERIC_FEATURES]
        .astype(np.float32)
        .to_numpy()
    )

    X_val_numeric = csr_matrix(
        val_fold[NUMERIC_FEATURES]
        .astype(np.float32)
        .to_numpy()
    )

    # 수치형 + One-Hot 범주형 결합
    X_train_xgb = hstack(
        [X_train_numeric, X_train_category],
        format="csr"
    )

    X_val_xgb = hstack(
        [X_val_numeric, X_val_category],
        format="csr"
    )

    print(f"\n{'=' * 70}")
    print(f"XGBoost 조합3 | Fold {fold_number}")
    print(f"{'=' * 70}")

    print(
        "Train 기간:",
        train_fold[TIME_COL].min(),
        "~",
        train_fold[TIME_COL].max()
    )

    print(
        "Validation 기간:",
        val_fold[TIME_COL].min(),
        "~",
        val_fold[TIME_COL].max()
    )

    print(f"Train 거래 수: {len(train_fold):,}")
    print(f"Validation 거래 수: {len(val_fold):,}")
    print(f"Train 이상거래 수: {positive_count:,}")
    print(
        f"Validation 이상거래 수: "
        f"{int(y_val_fold.sum()):,}"
    )
    print(
        f"scale_pos_weight: "
        f"{fold_scale_pos_weight:.6f}"
    )
    print(
        f"One-Hot 적용 후 변수 수: "
        f"{X_train_xgb.shape[1]}"
    )

    # --------------------------------------------------------
    # XGBoost 학습
    # --------------------------------------------------------

    model = XGBClassifier(
        **XGB_PARAMS,
        scale_pos_weight=fold_scale_pos_weight
    )

    training_start_time = time.perf_counter()

    model.fit(
        X_train_xgb,
        y_train_fold,
        eval_set=[(X_val_xgb, y_val_fold)],
        verbose=False
    )

    training_seconds = (
        time.perf_counter() - training_start_time
    )

    # --------------------------------------------------------
    # Validation 예측
    # --------------------------------------------------------

    validation_prediction_start = time.perf_counter()

    val_probability = model.predict_proba(
        X_val_xgb
    )[:, 1]

    validation_prediction_seconds = (
        time.perf_counter()
        - validation_prediction_start
    )

    # Train PR-AUC: 과적합 격차 확인용
    train_probability = model.predict_proba(
        X_train_xgb
    )[:, 1]

    train_pr_auc = average_precision_score(
        y_train_fold,
        train_probability
    )

    val_pr_auc = average_precision_score(
        y_val_fold,
        val_probability
    )

    train_val_gap = train_pr_auc - val_pr_auc

    # 공통 임계값 0.90 결과
    val_prediction_090 = (
        val_probability >= COMMON_THRESHOLD
    ).astype(np.int8)

    precision_090 = precision_score(
        y_val_fold,
        val_prediction_090,
        zero_division=0
    )

    recall_090 = recall_score(
        y_val_fold,
        val_prediction_090,
        zero_division=0
    )

    f1_090 = f1_score(
        y_val_fold,
        val_prediction_090,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_val_fold,
        val_prediction_090,
        labels=[0, 1]
    ).ravel()

    fold_total_seconds = (
        time.perf_counter() - fold_start_time
    )

    # --------------------------------------------------------
    # Fold 결과 저장
    # --------------------------------------------------------

    xgb_fold_results.append({
        "algorithm": "XGBoost",
        "combination": "조합3",
        "fold": fold_number,
        "train_count": len(train_fold),
        "validation_count": len(val_fold),
        "scale_pos_weight": fold_scale_pos_weight,
        "best_iteration": model.best_iteration,
        "train_pr_auc": train_pr_auc,
        "validation_pr_auc": val_pr_auc,
        "train_validation_gap": train_val_gap,
        "threshold": COMMON_THRESHOLD,
        "precision": precision_090,
        "recall": recall_090,
        "f1_score": f1_090,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "training_seconds": training_seconds,
        "validation_prediction_seconds":
            validation_prediction_seconds,
        "total_seconds": fold_total_seconds
    })

    # OOF 예측확률 저장
    xgb_oof_predictions.append(
        pd.DataFrame({
            "algorithm": "XGBoost",
            "combination": "조합3",
            "fold": fold_number,
            "original_index": val_idx,
            "y_true": y_val_fold.to_numpy(),
            "y_probability": val_probability
        })
    )

    # 모델과 인코더 저장
    xgb_fold_models.append({
        "fold": fold_number,
        "model": model,
        "encoder": encoder,
        "numeric_features": NUMERIC_FEATURES.copy(),
        "categorical_features":
            CATEGORICAL_FEATURES.copy()
    })

    print(f"✅ Fold {fold_number} 완료")
    print(f"Best iteration: {model.best_iteration}")
    print(f"Train PR-AUC: {train_pr_auc:.6f}")
    print(f"Validation PR-AUC: {val_pr_auc:.6f}")
    print(f"Train–Validation 격차: {train_val_gap:.6f}")
    print(f"임계값 0.90 Precision: {precision_090:.6f}")
    print(f"임계값 0.90 Recall: {recall_090:.6f}")
    print(f"임계값 0.90 F1: {f1_090:.6f}")
    print(f"FP: {fp:,} | FN: {fn:,}")
    print(f"학습 시간: {training_seconds / 60:.2f}분")

    del (
        train_fold,
        val_fold,
        X_train_category,
        X_val_category,
        X_train_numeric,
        X_val_numeric,
        X_train_xgb,
        X_val_xgb,
        train_probability,
        val_probability
    )

    gc.collect()


# ============================================================
# 6. Fold 결과 통합
# ============================================================

xgb_fold_results_df = pd.DataFrame(
    xgb_fold_results
)

xgb_oof_predictions_df = pd.concat(
    xgb_oof_predictions,
    ignore_index=True
)

total_elapsed_minutes = (
    time.perf_counter() - total_start_time
) / 60


# ============================================================
# 7. 결과 출력
# ============================================================

print("\n")
print("=" * 80)
print("XGBoost 조합3 시간순 3-Fold 결과")
print("=" * 80)

display(xgb_fold_results_df)

print(
    f"\n평균 Validation PR-AUC: "
    f"{xgb_fold_results_df['validation_pr_auc'].mean():.6f}"
)

print(
    f"Validation PR-AUC 표준편차: "
    f"{xgb_fold_results_df['validation_pr_auc'].std(ddof=1):.6f}"
)

print(
    f"평균 Train–Validation 격차: "
    f"{xgb_fold_results_df['train_validation_gap'].mean():.6f}"
)

print(
    f"평균 Best iteration: "
    f"{xgb_fold_results_df['best_iteration'].mean():.2f}"
)

print(
    f"전체 실행 시간: "
    f"{total_elapsed_minutes:.2f}분"
)

print(
    f"OOF 예측 저장 행 수: "
    f"{len(xgb_oof_predictions_df):,}"
)

✅ XGBoost 조합3 학습 준비 완료
Fold 수: 3
원본 변수 수: 11

XGBoost 조합3 | Fold 1
Train 기간: 2019-01-01 00:00:18 ~ 2019-10-03 07:35:11
Validation 기간: 2019-10-03 07:35:47 ~ 2019-12-18 17:06:47
Train 거래 수: 648,337
Validation 거래 수: 216,113
Train 이상거래 수: 3,827
Validation 이상거래 수: 1,091
scale_pos_weight: 168.411288
One-Hot 적용 후 변수 수: 24
✅ Fold 1 완료
Best iteration: 1432
Train PR-AUC: 0.999780
Validation PR-AUC: 0.971303
Train–Validation 격차: 0.028477
임계값 0.90 Precision: 0.957854
임계값 0.90 Recall: 0.916590
임계값 0.90 F1: 0.936768
FP: 44 | FN: 91
학습 시간: 6.12분

XGBoost 조합3 | Fold 2
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-18 17:06:47
Validation 기간: 2019-12-18 17:07:05 ~ 2020-03-24 15:14:25
Train 거래 수: 864,450
Validation 거래 수: 216,112
Train 이상거래 수: 4,918
Validation 이상거래 수: 1,363
scale_pos_weight: 174.772672
One-Hot 적용 후 변수 수: 24
✅ Fold 2 완료
Best iteration: 1227
Train PR-AUC: 0.999019
Validation PR-AUC: 0.978763
Train–Validation 격차: 0.020256
임계값 0.90 Precision: 0.945685
임계값 0.90 Recall: 0.932502
임계값 0.90 F1: 0.939047


,algorithm,combination,fold,train_count,validation_count,scale_pos_weight,best_iteration,train_pr_auc,validation_pr_auc,train_validation_gap,...,precision,recall,f1_score,tn,fp,fn,tp,training_seconds,validation_prediction_seconds,total_seconds
0,XGBoost,조합3,1,648337,216113,168.411288,1432,0.999780,0.971303,0.028477,...,0.957854,0.916590,0.936768,214978,44,91,1000,367.318595,3.488799,385.437450
1,XGBoost,조합3,2,864450,216112,174.772672,1227,0.999019,0.978763,0.020256,...,0.945685,0.932502,0.939047,214676,73,92,1271,279.997903,3.851426,301.350435
2,XGBoost,조합3,3,1080562,216113,171.036618,1124,0.998052,0.975511,0.022541,...,0.931507,0.943673,0.937551,214803,85,69,1156,317.665167,3.020278,340.772359



평균 Validation PR-AUC: 0.975192
Validation PR-AUC 표준편차: 0.003740
평균 Train–Validation 격차: 0.023758
평균 Best iteration: 1261.00
전체 실행 시간: 17.14분
OOF 예측 저장 행 수: 648,338


xgboost 시간순 3-fold 검증, 기존 xgboost 최적 파라미터 적용 완료.

1차 비교 결과

" 성능은 사실상 동률이고, 효율성까지 고려하면 lightGBM이 우세. "

Fold	LightGBM PR-AUC   	XGBoost PR-AUC	   차이(XGB−LGBM)
Fold 1	  0.970885          	0.971303	    +0.000418
Fold 2	  0.978991	            0.978763     	−0.000228
Fold 3	  0.974993	            0.975511     	+0.000518
평균	  0.974956	            0.975192     	+0.000236
표준편차  	0.004053         	0.003740      	−0.000313

------------------------------------------------------------------------------

lightGBM의 OOF 예측 확률을 동일한 3-Fold에서 다시 수집

In [5]:
# ============================================================
# STEP 2. LightGBM 조합3 OOF 예측확률 수집
# - XGBoost와 동일한 행 기준 3-Fold
# - LightGBM에서 선택된 최종 설정 사용
# ============================================================

import gc
import time
import numpy as np
import pandas as pd

from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation
)

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# ============================================================
# 1. LightGBM 최종 설정
# ============================================================

LGBM_COMBINATION3_FEATURES = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",
    "high_speed"
]

LGBM_CATEGORICAL_FEATURES = ["category"]

LGBM_THRESHOLD_REFERENCE = 0.90

LGBM_PARAMS = {
    "objective": "binary",
    "n_estimators": 3000,
    "learning_rate": 0.03,

    # 기존 3가지 설정 중 최종 선택값
    "num_leaves": 15,
    "min_child_samples": 100,

    "max_bin": 255,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1
}


# ============================================================
# 2. 변수 확인
# ============================================================

if "df" not in globals():
    raise NameError("df가 없습니다.")

if "time_folds" not in globals():
    raise NameError("time_folds가 없습니다.")

if "xgb_oof_predictions_df" not in globals():
    raise NameError(
        "xgb_oof_predictions_df가 없습니다. "
        "앞의 XGBoost 3-Fold 코드를 먼저 실행해주세요."
    )

missing_columns = [
    column
    for column in LGBM_COMBINATION3_FEATURES + [TARGET]
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "다음 변수가 없습니다: "
        + ", ".join(missing_columns)
    )

print("✅ LightGBM OOF 생성 준비 완료")


# ============================================================
# 3. 결과 저장 공간
# ============================================================

lgbm_fold_results = []
lgbm_oof_predictions = []
lgbm_fold_models = []

lgbm_total_start = time.perf_counter()


# ============================================================
# 4. 동일한 시간순 3-Fold 학습
# ============================================================

for fold_info in time_folds:

    fold_number = fold_info["fold"]
    fold_start = time.perf_counter()

    train_idx = fold_info["train_idx"]
    val_idx = fold_info["val_idx"]

    X_train_lgbm = (
        df.iloc[train_idx]
        [LGBM_COMBINATION3_FEATURES]
        .copy()
        .reset_index(drop=True)
    )

    X_val_lgbm = (
        df.iloc[val_idx]
        [LGBM_COMBINATION3_FEATURES]
        .copy()
        .reset_index(drop=True)
    )

    y_train_lgbm = (
        df.iloc[train_idx][TARGET]
        .astype(np.int8)
        .reset_index(drop=True)
    )

    y_val_lgbm = (
        df.iloc[val_idx][TARGET]
        .astype(np.int8)
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Category 범주를 Fold Train 기준으로 통일
    # --------------------------------------------------------

    for column in LGBM_CATEGORICAL_FEATURES:

        train_categories = (
            X_train_lgbm[column]
            .astype("string")
            .fillna("missing")
            .unique()
            .tolist()
        )

        X_train_lgbm[column] = pd.Categorical(
            X_train_lgbm[column]
            .astype("string")
            .fillna("missing"),
            categories=train_categories
        )

        X_val_lgbm[column] = pd.Categorical(
            X_val_lgbm[column]
            .astype("string")
            .fillna("missing"),
            categories=train_categories
        )

    # --------------------------------------------------------
    # Fold Train 기준 불균형 가중치
    # --------------------------------------------------------

    negative_count = int((y_train_lgbm == 0).sum())
    positive_count = int((y_train_lgbm == 1).sum())

    fold_scale_pos_weight = (
        negative_count / positive_count
    )

    print(f"\n{'=' * 70}")
    print(f"LightGBM 조합3 | Fold {fold_number}")
    print(f"{'=' * 70}")

    print(f"Train 거래 수: {len(X_train_lgbm):,}")
    print(f"Validation 거래 수: {len(X_val_lgbm):,}")
    print(f"Train 이상거래 수: {positive_count:,}")
    print(
        f"Validation 이상거래 수: "
        f"{int(y_val_lgbm.sum()):,}"
    )
    print(
        f"scale_pos_weight: "
        f"{fold_scale_pos_weight:.6f}"
    )

    # --------------------------------------------------------
    # 모델 학습
    # --------------------------------------------------------

    model = LGBMClassifier(
        **LGBM_PARAMS,
        scale_pos_weight=fold_scale_pos_weight
    )

    training_start = time.perf_counter()

    model.fit(
        X_train_lgbm,
        y_train_lgbm,

        eval_set=[
            (X_val_lgbm, y_val_lgbm)
        ],

        eval_metric="average_precision",

        categorical_feature=
            LGBM_CATEGORICAL_FEATURES,

        callbacks=[
            early_stopping(
                stopping_rounds=100,
                first_metric_only=True,
                verbose=False
            ),
            log_evaluation(period=0)
        ]
    )

    training_seconds = (
        time.perf_counter() - training_start
    )

    # --------------------------------------------------------
    # Validation 확률
    # --------------------------------------------------------

    prediction_start = time.perf_counter()

    val_probability = model.predict_proba(
        X_val_lgbm,
        num_iteration=model.best_iteration_
    )[:, 1]

    prediction_seconds = (
        time.perf_counter() - prediction_start
    )

    val_pr_auc = average_precision_score(
        y_val_lgbm,
        val_probability
    )

    # 임계값 0.90은 참고용
    val_prediction_090 = (
        val_probability >= LGBM_THRESHOLD_REFERENCE
    ).astype(np.int8)

    precision_090 = precision_score(
        y_val_lgbm,
        val_prediction_090,
        zero_division=0
    )

    recall_090 = recall_score(
        y_val_lgbm,
        val_prediction_090,
        zero_division=0
    )

    f1_090 = f1_score(
        y_val_lgbm,
        val_prediction_090,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_val_lgbm,
        val_prediction_090,
        labels=[0, 1]
    ).ravel()

    fold_total_seconds = (
        time.perf_counter() - fold_start
    )

    # --------------------------------------------------------
    # Fold 결과 저장
    # --------------------------------------------------------

    lgbm_fold_results.append({
        "algorithm": "LightGBM",
        "combination": "조합3",
        "fold": fold_number,
        "scale_pos_weight": fold_scale_pos_weight,
        "best_iteration": model.best_iteration_,
        "validation_pr_auc": val_pr_auc,
        "threshold": LGBM_THRESHOLD_REFERENCE,
        "precision": precision_090,
        "recall": recall_090,
        "f1_score": f1_090,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "training_seconds": training_seconds,
        "validation_prediction_seconds":
            prediction_seconds,
        "total_seconds": fold_total_seconds
    })

    # OOF 예측확률 저장
    lgbm_oof_predictions.append(
        pd.DataFrame({
            "algorithm": "LightGBM",
            "combination": "조합3",
            "fold": fold_number,
            "original_index": val_idx,
            "y_true": y_val_lgbm.to_numpy(),
            "y_probability": val_probability
        })
    )

    lgbm_fold_models.append({
        "fold": fold_number,
        "model": model
    })

    print(f"✅ Fold {fold_number} 완료")
    print(f"Best iteration: {model.best_iteration_}")
    print(f"Validation PR-AUC: {val_pr_auc:.6f}")
    print(f"임계값 0.90 Precision: {precision_090:.6f}")
    print(f"임계값 0.90 Recall: {recall_090:.6f}")
    print(f"임계값 0.90 F1: {f1_090:.6f}")
    print(f"FP: {fp:,} | FN: {fn:,}")
    print(f"학습 시간: {training_seconds / 60:.2f}분")

    del (
        X_train_lgbm,
        X_val_lgbm,
        y_train_lgbm,
        y_val_lgbm,
        val_probability
    )

    gc.collect()


# ============================================================
# 5. 결과 통합
# ============================================================

lgbm_fold_results_df = pd.DataFrame(
    lgbm_fold_results
)

lgbm_oof_predictions_df = pd.concat(
    lgbm_oof_predictions,
    ignore_index=True
)

lgbm_total_minutes = (
    time.perf_counter() - lgbm_total_start
) / 60


# ============================================================
# 6. XGBoost와 OOF 데이터 일치 여부 확인
# ============================================================

if len(lgbm_oof_predictions_df) != len(
    xgb_oof_predictions_df
):
    raise ValueError(
        "LightGBM과 XGBoost의 OOF 행 수가 다릅니다."
    )

if not np.array_equal(
    lgbm_oof_predictions_df["original_index"].to_numpy(),
    xgb_oof_predictions_df["original_index"].to_numpy()
):
    raise ValueError(
        "LightGBM과 XGBoost의 Validation 행 구성이 다릅니다."
    )

if not np.array_equal(
    lgbm_oof_predictions_df["y_true"].to_numpy(),
    xgb_oof_predictions_df["y_true"].to_numpy()
):
    raise ValueError(
        "LightGBM과 XGBoost의 OOF 정답이 다릅니다."
    )


# ============================================================
# 7. 출력
# ============================================================

print("\n")
print("=" * 80)
print("LightGBM 조합3 OOF 예측 수집 완료")
print("=" * 80)

display(lgbm_fold_results_df)

print(
    f"\n평균 Validation PR-AUC: "
    f"{lgbm_fold_results_df['validation_pr_auc'].mean():.6f}"
)

print(
    f"PR-AUC 표준편차: "
    f"{lgbm_fold_results_df['validation_pr_auc'].std(ddof=1):.6f}"
)

print(
    f"전체 실행 시간: "
    f"{lgbm_total_minutes:.2f}분"
)

print(
    f"LightGBM OOF 행 수: "
    f"{len(lgbm_oof_predictions_df):,}"
)

print(
    f"XGBoost OOF 행 수: "
    f"{len(xgb_oof_predictions_df):,}"
)

print("\n✅ 두 알고리즘의 OOF Validation 행이 동일합니다.")
print("✅ 다음 단계에서 임계값을 공정하게 비교할 수 있습니다.")

✅ LightGBM OOF 생성 준비 완료

LightGBM 조합3 | Fold 1
Train 거래 수: 648,337
Validation 거래 수: 216,113
Train 이상거래 수: 3,827
Validation 이상거래 수: 1,091
scale_pos_weight: 168.411288


c:\Users\splen\OneDrive\Desktop\BDAI_\BDAI\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


✅ Fold 1 완료
Best iteration: 1039
Validation PR-AUC: 0.970885
임계값 0.90 Precision: 0.892826
임계값 0.90 Recall: 0.946838
임계값 0.90 F1: 0.919039
FP: 124 | FN: 58
학습 시간: 1.38분

LightGBM 조합3 | Fold 2
Train 거래 수: 864,450
Validation 거래 수: 216,112
Train 이상거래 수: 4,918
Validation 이상거래 수: 1,363
scale_pos_weight: 174.772672


c:\Users\splen\OneDrive\Desktop\BDAI_\BDAI\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


✅ Fold 2 완료
Best iteration: 1312
Validation PR-AUC: 0.978991
임계값 0.90 Precision: 0.919149
임계값 0.90 Recall: 0.950844
임계값 0.90 F1: 0.934728
FP: 114 | FN: 67
학습 시간: 1.80분

LightGBM 조합3 | Fold 3
Train 거래 수: 1,080,562
Validation 거래 수: 216,113
Train 이상거래 수: 6,281
Validation 이상거래 수: 1,225
scale_pos_weight: 171.036618


c:\Users\splen\OneDrive\Desktop\BDAI_\BDAI\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


✅ Fold 3 완료
Best iteration: 1236
Validation PR-AUC: 0.974993
임계값 0.90 Precision: 0.905616
임계값 0.90 Recall: 0.947755
임계값 0.90 F1: 0.926207
FP: 121 | FN: 64
학습 시간: 1.65분


LightGBM 조합3 OOF 예측 수집 완료


,algorithm,combination,fold,scale_pos_weight,best_iteration,validation_pr_auc,threshold,precision,recall,f1_score,tn,fp,fn,tp,training_seconds,validation_prediction_seconds,total_seconds
0,LightGBM,조합3,1,168.411288,1039,0.970885,0.9,0.892826,0.946838,0.919039,214898,124,58,1033,82.847041,9.200981,92.930856
1,LightGBM,조합3,2,174.772672,1312,0.978991,0.9,0.919149,0.950844,0.934728,214635,114,67,1296,107.740083,11.123534,120.883913
2,LightGBM,조합3,3,171.036618,1236,0.974993,0.9,0.905616,0.947755,0.926207,214767,121,64,1161,98.780209,9.152804,109.828518



평균 Validation PR-AUC: 0.974956
PR-AUC 표준편차: 0.004053
전체 실행 시간: 5.40분
LightGBM OOF 행 수: 648,338
XGBoost OOF 행 수: 648,338

✅ 두 알고리즘의 OOF Validation 행이 동일합니다.
✅ 다음 단계에서 임계값을 공정하게 비교할 수 있습니다.


---

아래 코드는 다음 내용을 한 번에 비교.

- 통합 OOF PR-AUC
- 각 알고리즘의 F1 최대 임계값
- 해당 임계값의 Precision·Recall·F1·FP·FN
- 동일 목표 Recall 0.93을 만족하면서 Precision이 가장 높은 임계값과 결과

In [6]:
# ============================================================
# STEP 3. LightGBM vs XGBoost OOF 임계값 최적화
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# ============================================================
# 1. 비교 기준
# ============================================================

TARGET_RECALL = 0.93


# ============================================================
# 2. 평가 함수
# ============================================================

def calculate_threshold_metrics(
    algorithm,
    y_true,
    y_probability,
    threshold,
    selection_method
):

    y_prediction = (
        y_probability >= threshold
    ).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_prediction,
        labels=[0, 1]
    ).ravel()

    return {
        "algorithm": algorithm,
        "selection_method": selection_method,
        "threshold": threshold,

        "pr_auc": average_precision_score(
            y_true,
            y_probability
        ),

        "precision": precision_score(
            y_true,
            y_prediction,
            zero_division=0
        ),

        "recall": recall_score(
            y_true,
            y_prediction,
            zero_division=0
        ),

        "f1_score": f1_score(
            y_true,
            y_prediction,
            zero_division=0
        ),

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,

        "actual_positive": int((y_true == 1).sum()),
        "predicted_positive": int(y_prediction.sum())
    }


# ============================================================
# 3. F1 최대 임계값 탐색 함수
# ============================================================

def find_best_f1_threshold(
    y_true,
    y_probability
):

    precision_values, recall_values, thresholds = (
        precision_recall_curve(
            y_true,
            y_probability
        )
    )

    # 마지막 precision/recall 값에는 대응 임계값이 없으므로 제외
    precision_values = precision_values[:-1]
    recall_values = recall_values[:-1]

    f1_values = np.divide(
        2 * precision_values * recall_values,
        precision_values + recall_values,
        out=np.zeros_like(precision_values),
        where=(
            precision_values + recall_values
        ) != 0
    )

    best_index = np.argmax(f1_values)

    return float(thresholds[best_index])


# ============================================================
# 4. 목표 Recall 기준 임계값 탐색 함수
# ============================================================

def find_target_recall_threshold(
    y_true,
    y_probability,
    target_recall=0.93
):

    precision_values, recall_values, thresholds = (
        precision_recall_curve(
            y_true,
            y_probability
        )
    )

    precision_values = precision_values[:-1]
    recall_values = recall_values[:-1]

    valid_indices = np.where(
        recall_values >= target_recall
    )[0]

    if len(valid_indices) == 0:
        raise ValueError(
            f"Recall {target_recall:.2f}를 "
            "만족하는 임계값이 없습니다."
        )

    # 목표 Recall을 만족하는 후보 중 Precision 최대
    valid_precision = precision_values[
        valid_indices
    ]

    maximum_precision = valid_precision.max()

    best_candidates = valid_indices[
        np.isclose(
            valid_precision,
            maximum_precision
        )
    ]

    # Precision이 같으면 더 높은 임계값 선택
    best_index = best_candidates[
        np.argmax(thresholds[best_candidates])
    ]

    return float(thresholds[best_index])


# ============================================================
# 5. 알고리즘별 OOF 데이터 준비
# ============================================================

oof_datasets = {
    "LightGBM": (
        lgbm_oof_predictions_df
        .sort_values("original_index")
        .reset_index(drop=True)
    ),

    "XGBoost": (
        xgb_oof_predictions_df
        .sort_values("original_index")
        .reset_index(drop=True)
    )
}


# ============================================================
# 6. OOF 행과 정답 재확인
# ============================================================

lgbm_check = oof_datasets["LightGBM"]
xgb_check = oof_datasets["XGBoost"]

if not np.array_equal(
    lgbm_check["original_index"].to_numpy(),
    xgb_check["original_index"].to_numpy()
):
    raise ValueError(
        "두 알고리즘의 OOF 행이 서로 다릅니다."
    )

if not np.array_equal(
    lgbm_check["y_true"].to_numpy(),
    xgb_check["y_true"].to_numpy()
):
    raise ValueError(
        "두 알고리즘의 OOF 정답이 서로 다릅니다."
    )

print("✅ OOF 비교 대상 일치 확인 완료")


# ============================================================
# 7. 알고리즘별 임계값 최적화
# ============================================================

oof_comparison_results = []

for algorithm, prediction_df in oof_datasets.items():

    y_true = prediction_df[
        "y_true"
    ].to_numpy()

    y_probability = prediction_df[
        "y_probability"
    ].to_numpy()

    # --------------------------------------------------------
    # A. F1 최대 임계값
    # --------------------------------------------------------

    best_f1_threshold = find_best_f1_threshold(
        y_true,
        y_probability
    )

    oof_comparison_results.append(
        calculate_threshold_metrics(
            algorithm=algorithm,
            y_true=y_true,
            y_probability=y_probability,
            threshold=best_f1_threshold,
            selection_method="F1 최대"
        )
    )

    # --------------------------------------------------------
    # B. 동일 목표 Recall 기준
    # --------------------------------------------------------

    target_recall_threshold = (
        find_target_recall_threshold(
            y_true,
            y_probability,
            target_recall=TARGET_RECALL
        )
    )

    oof_comparison_results.append(
        calculate_threshold_metrics(
            algorithm=algorithm,
            y_true=y_true,
            y_probability=y_probability,
            threshold=target_recall_threshold,
            selection_method=(
                f"Recall≥{TARGET_RECALL:.2f}"
            )
        )
    )


# ============================================================
# 8. 결과표
# ============================================================

oof_comparison_df = pd.DataFrame(
    oof_comparison_results
)

oof_comparison_df = oof_comparison_df[
    [
        "selection_method",
        "algorithm",
        "threshold",
        "pr_auc",
        "precision",
        "recall",
        "f1_score",
        "fp",
        "fn",
        "tp",
        "tn",
        "actual_positive",
        "predicted_positive"
    ]
]

print("\n")
print("=" * 90)
print("LightGBM vs XGBoost OOF 임계값 비교")
print("=" * 90)

display(
    oof_comparison_df.style.format({
        "threshold": "{:.6f}",
        "pr_auc": "{:.6f}",
        "precision": "{:.6f}",
        "recall": "{:.6f}",
        "f1_score": "{:.6f}"
    })
)


# ============================================================
# 9. 조건별 승자 출력
# ============================================================

for selection_method in (
    oof_comparison_df["selection_method"].unique()
):

    comparison_part = oof_comparison_df[
        oof_comparison_df["selection_method"]
        == selection_method
    ]

    print(f"\n===== {selection_method} =====")

    if selection_method == "F1 최대":

        winner = comparison_part.loc[
            comparison_part["f1_score"].idxmax()
        ]

        print(
            f"F1 우세 모델: "
            f"{winner['algorithm']}"
        )
        print(
            f"F1: "
            f"{winner['f1_score']:.6f}"
        )

    else:

        winner = comparison_part.sort_values(
            by=["precision", "fp"],
            ascending=[False, True]
        ).iloc[0]

        print(
            f"동일 Recall 기준 우세 모델: "
            f"{winner['algorithm']}"
        )
        print(
            f"Precision: "
            f"{winner['precision']:.6f}"
        )
        print(
            f"FP: {int(winner['fp']):,}"
        )
        print(
            f"FN: {int(winner['fn']):,}"
        )

✅ OOF 비교 대상 일치 확인 완료


LightGBM vs XGBoost OOF 임계값 비교


,selection_method,algorithm,threshold,pr_auc,precision,recall,f1_score,fp,fn,tp,tn,actual_positive,predicted_positive
0,F1 최대,LightGBM,0.971298,0.975170,0.961375,0.920087,0.940278,136,294,3385,644523,3679,3521
1,Recall≥0.93,LightGBM,0.958989,0.975170,0.948974,0.930144,0.939465,184,257,3422,644475,3679,3606
2,F1 최대,XGBoost,0.946612,0.975038,0.966208,0.917097,0.941012,118,305,3374,644541,3679,3492
3,Recall≥0.93,XGBoost,0.909143,0.975038,0.946873,0.930144,0.938434,192,257,3422,644467,3679,3614



===== F1 최대 =====
F1 우세 모델: XGBoost
F1: 0.941012

===== Recall≥0.93 =====
동일 Recall 기준 우세 모델: LightGBM
Precision: 0.948974
FP: 184
FN: 257


결과 잘 나왔고, lightGBM 선택 근거가 더 강해짐.

< OOF 비교결과 >

1. 통합 PR-AUC

차이는 약 0.013%p로 사실상 동률입니다. 참고로 앞에서는 Fold별 PR-AUC의 단순 평균을 비교했고, 지금은 모든 Validation 예측을 합친 OOF PR-AUC라 순위가 미세하게 달라진 것입니다. 오류가 아닙니다.

2. 각 모델의 F1 최대 기준

XGBoost의 F1이 0.000734 높지만 매우 작은 차이입니다.

XGBoost: 오탐 FP가 18건 적음
LightGBM: 미탐 FN이 11건 적음
이상거래 탐지에서 FN을 더 중요하게 본다면 LightGBM 쪽이 낫습니다.

3. 동일 Recall 0.93 기준

이 비교가 가장 중요합니다. 두 모델이 똑같이 이상거래 3,422건을 잡고 257건을 놓쳤는데:

LightGBM FP: 184건
XGBoost FP: 192건
LightGBM이 정상거래 오탐을 8건 덜 발생
LightGBM Precision도 약 0.0021 높음

따라서 같은 Recall에서는 LightGBM이 우세합니다.

---

마지막으로 Fold별 최적 임계값이 얼마나 흔들리는지, 최적 임계값 주변의 민감도 확인하기.

- Fold마다 F1 최적 임계값이 얼마나 달라지는지
- Fold마다 Recall 0.93을 만족하는 임계값이 얼마나 달라지는지
- 통합 OOF 최적 임계값을 ±0.01, ±0.02 변경했을 때 성능이 급격히 바뀌는지

In [9]:
# ============================================================
# STEP 4. LightGBM vs XGBoost 임계값 안정성 비교
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# ============================================================
# 1. 설정
# ============================================================

TARGET_RECALL = 0.93

THRESHOLD_OFFSETS = [
    -0.02,
    -0.01,
    0.00,
    0.01,
    0.02
]


# ============================================================
# 2. Fold별 최적 임계값 계산
# ============================================================

fold_threshold_results = []

for algorithm, prediction_df in oof_datasets.items():

    for fold_number in sorted(
        prediction_df["fold"].unique()
    ):

        fold_df = prediction_df[
            prediction_df["fold"] == fold_number
        ]

        y_true = fold_df[
            "y_true"
        ].to_numpy()

        y_probability = fold_df[
            "y_probability"
        ].to_numpy()

        # F1 최대 임계값
        best_f1_threshold = (
            find_best_f1_threshold(
                y_true,
                y_probability
            )
        )

        best_f1_metrics = (
            calculate_threshold_metrics(
                algorithm=algorithm,
                y_true=y_true,
                y_probability=y_probability,
                threshold=best_f1_threshold,
                selection_method="Fold별 F1 최대"
            )
        )

        # Recall 0.93 이상을 만족하면서 Precision 최대
        target_recall_threshold = (
            find_target_recall_threshold(
                y_true,
                y_probability,
                target_recall=TARGET_RECALL
            )
        )

        target_recall_metrics = (
            calculate_threshold_metrics(
                algorithm=algorithm,
                y_true=y_true,
                y_probability=y_probability,
                threshold=target_recall_threshold,
                selection_method=(
                    f"Fold별 Recall≥{TARGET_RECALL:.2f}"
                )
            )
        )

        for result in [
            best_f1_metrics,
            target_recall_metrics
        ]:

            result["fold"] = fold_number

            fold_threshold_results.append(
                result
            )


fold_threshold_df = pd.DataFrame(
    fold_threshold_results
)


# ============================================================
# 3. Fold별 임계값 출력
# ============================================================

fold_threshold_display = fold_threshold_df[
    [
        "selection_method",
        "algorithm",
        "fold",
        "threshold",
        "precision",
        "recall",
        "f1_score",
        "fp",
        "fn"
    ]
].sort_values(
    [
        "selection_method",
        "algorithm",
        "fold"
    ]
)

print("\n")
print("=" * 90)
print("Fold별 최적 임계값")
print("=" * 90)

display(
    fold_threshold_display.style.format({
        "threshold": "{:.6f}",
        "precision": "{:.6f}",
        "recall": "{:.6f}",
        "f1_score": "{:.6f}"
    })
)


# ============================================================
# 4. Fold별 임계값 안정성 요약
# ============================================================

threshold_stability_df = (
    fold_threshold_df
    .groupby(
        [
            "selection_method",
            "algorithm"
        ],
        as_index=False
    )
    .agg(
        mean_threshold=(
            "threshold",
            "mean"
        ),

        std_threshold=(
            "threshold",
            lambda x: x.std(ddof=1)
        ),

        min_threshold=(
            "threshold",
            "min"
        ),

        max_threshold=(
            "threshold",
            "max"
        ),

        mean_f1=(
            "f1_score",
            "mean"
        ),

        std_f1=(
            "f1_score",
            lambda x: x.std(ddof=1)
        ),

        mean_precision=(
            "precision",
            "mean"
        ),

        mean_recall=(
            "recall",
            "mean"
        ),

        total_fp=(
            "fp",
            "sum"
        ),

        total_fn=(
            "fn",
            "sum"
        )
    )
)

threshold_stability_df[
    "threshold_range"
] = (
    threshold_stability_df["max_threshold"]
    - threshold_stability_df["min_threshold"]
)

print("\n")
print("=" * 90)
print("Fold별 임계값 안정성 요약")
print("=" * 90)

display(
    threshold_stability_df.style.format({
        "mean_threshold": "{:.6f}",
        "std_threshold": "{:.6f}",
        "min_threshold": "{:.6f}",
        "max_threshold": "{:.6f}",
        "threshold_range": "{:.6f}",
        "mean_f1": "{:.6f}",
        "std_f1": "{:.6f}",
        "mean_precision": "{:.6f}",
        "mean_recall": "{:.6f}"
    })
)


# ============================================================
# 5. 통합 OOF F1 최적 임계값 불러오기
# ============================================================

pooled_f1_thresholds = {}

for algorithm in oof_datasets.keys():

    selected_row = oof_comparison_df[
        (
            oof_comparison_df["algorithm"]
            == algorithm
        )
        &
        (
            oof_comparison_df["selection_method"]
            == "F1 최대"
        )
    ]

    if len(selected_row) != 1:
        raise ValueError(
            f"{algorithm}의 OOF F1 최적 임계값을 "
            "찾을 수 없습니다."
        )

    pooled_f1_thresholds[algorithm] = float(
        selected_row.iloc[0]["threshold"]
    )


# ============================================================
# 6. 최적 임계값 주변 민감도 분석
# ============================================================

threshold_sensitivity_results = []

for algorithm, prediction_df in oof_datasets.items():

    y_true = prediction_df[
        "y_true"
    ].to_numpy()

    y_probability = prediction_df[
        "y_probability"
    ].to_numpy()

    optimal_threshold = (
        pooled_f1_thresholds[algorithm]
    )

    for offset in THRESHOLD_OFFSETS:

        test_threshold = np.clip(
            optimal_threshold + offset,
            0,
            1
        )

        y_prediction = (
            y_probability >= test_threshold
        ).astype(np.int8)

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_prediction,
            labels=[0, 1]
        ).ravel()

        threshold_sensitivity_results.append({
            "algorithm": algorithm,
            "optimal_threshold":
                optimal_threshold,
            "offset": offset,
            "test_threshold":
                test_threshold,

            "precision": precision_score(
                y_true,
                y_prediction,
                zero_division=0
            ),

            "recall": recall_score(
                y_true,
                y_prediction,
                zero_division=0
            ),

            "f1_score": f1_score(
                y_true,
                y_prediction,
                zero_division=0
            ),

            "fp": fp,
            "fn": fn,
            "tp": tp,
            "tn": tn
        })


threshold_sensitivity_df = pd.DataFrame(
    threshold_sensitivity_results
)


# ============================================================
# 7. 임계값 주변 성능 출력
# ============================================================

print("\n")
print("=" * 90)
print("통합 OOF 최적 임계값 주변 민감도")
print("=" * 90)

display(
    threshold_sensitivity_df.style.format({
        "optimal_threshold": "{:.6f}",
        "offset": "{:+.2f}",
        "test_threshold": "{:.6f}",
        "precision": "{:.6f}",
        "recall": "{:.6f}",
        "f1_score": "{:.6f}"
    })
)


# ============================================================
# 8. 민감도 요약
# ============================================================

sensitivity_summary_results = []

for algorithm in oof_datasets.keys():

    algorithm_result = (
        threshold_sensitivity_df[
            threshold_sensitivity_df[
                "algorithm"
            ] == algorithm
        ]
    )

    center_result = algorithm_result[
        np.isclose(
            algorithm_result["offset"],
            0.00
        )
    ].iloc[0]

    f1_min = algorithm_result[
        "f1_score"
    ].min()

    recall_min = algorithm_result[
        "recall"
    ].min()

    recall_max = algorithm_result[
        "recall"
    ].max()

    fp_min = algorithm_result["fp"].min()
    fp_max = algorithm_result["fp"].max()

    fn_min = algorithm_result["fn"].min()
    fn_max = algorithm_result["fn"].max()

    sensitivity_summary_results.append({
        "algorithm": algorithm,

        "optimal_threshold":
            center_result["test_threshold"],

        "optimal_f1":
            center_result["f1_score"],

        "minimum_f1_around_threshold":
            f1_min,

        "maximum_f1_drop":
            center_result["f1_score"] - f1_min,

        "recall_range":
            recall_max - recall_min,

        "fp_change_range":
            fp_max - fp_min,

        "fn_change_range":
            fn_max - fn_min
    })


sensitivity_summary_df = pd.DataFrame(
    sensitivity_summary_results
)

print("\n")
print("=" * 90)
print("임계값 민감도 최종 요약")
print("=" * 90)

display(
    sensitivity_summary_df.style.format({
        "optimal_threshold": "{:.6f}",
        "optimal_f1": "{:.6f}",
        "minimum_f1_around_threshold": "{:.6f}",
        "maximum_f1_drop": "{:.6f}",
        "recall_range": "{:.6f}"
    })
)



Fold별 최적 임계값


,selection_method,algorithm,fold,threshold,precision,recall,f1_score,fp,fn
0,Fold별 F1 최대,LightGBM,1,0.971298,0.960501,0.913841,0.936590,41,94
2,Fold별 F1 최대,LightGBM,2,0.953029,0.954170,0.931768,0.942836,61,93
4,Fold별 F1 최대,LightGBM,3,0.967904,0.957322,0.933878,0.945455,51,81
6,Fold별 F1 최대,XGBoost,1,0.887158,0.954502,0.923006,0.938490,48,84
8,Fold별 F1 최대,XGBoost,2,0.943307,0.964751,0.923698,0.943778,46,104
10,Fold별 F1 최대,XGBoost,3,0.948304,0.960438,0.931429,0.945711,47,84
1,Fold별 Recall≥0.93,LightGBM,1,0.953965,0.938945,0.930339,0.934622,66,76
3,Fold별 Recall≥0.93,LightGBM,2,0.954312,0.954853,0.931034,0.942793,60,94
5,Fold별 Recall≥0.93,LightGBM,3,0.967904,0.957322,0.933878,0.945455,51,81
7,Fold별 Recall≥0.93,XGBoost,1,0.807764,0.933945,0.933089,0.933517,72,73




Fold별 임계값 안정성 요약


,selection_method,algorithm,mean_threshold,std_threshold,min_threshold,max_threshold,mean_f1,std_f1,mean_precision,mean_recall,total_fp,total_fn,threshold_range
0,Fold별 F1 최대,LightGBM,0.964077,0.009717,0.953029,0.971298,0.941627,0.004554,0.957331,0.926495,153,268,0.018268
1,Fold별 F1 최대,XGBoost,0.926256,0.033952,0.887158,0.948304,0.942660,0.003738,0.959897,0.926044,141,272,0.061147
2,Fold별 Recall≥0.93,LightGBM,0.958727,0.007949,0.953965,0.967904,0.940957,0.005645,0.950374,0.931750,177,251,0.013938
3,Fold별 Recall≥0.93,XGBoost,0.890342,0.073432,0.807764,0.948304,0.940193,0.006179,0.949017,0.931606,182,252,0.140540




통합 OOF 최적 임계값 주변 민감도


,algorithm,optimal_threshold,offset,test_threshold,precision,recall,f1_score,fp,fn,tp,tn
0,LightGBM,0.971298,-0.02,0.951298,0.943163,0.933678,0.938396,207,244,3435,644452
1,LightGBM,0.971298,-0.01,0.961298,0.950710,0.927970,0.939202,177,265,3414,644482
2,LightGBM,0.971298,+0.00,0.971298,0.961375,0.920087,0.940278,136,294,3385,644523
3,LightGBM,0.971298,+0.01,0.981298,0.973115,0.905137,0.937896,92,349,3330,644567
4,LightGBM,0.971298,+0.02,0.991298,0.984620,0.870073,0.923810,50,478,3201,644609
5,XGBoost,0.946612,-0.02,0.926612,0.954813,0.924708,0.939519,161,277,3402,644498
6,XGBoost,0.946612,-0.01,0.936612,0.961157,0.921446,0.940883,137,289,3390,644522
7,XGBoost,0.946612,+0.00,0.946612,0.966208,0.917097,0.941012,118,305,3374,644541
8,XGBoost,0.946612,+0.01,0.956612,0.969890,0.910574,0.939296,104,329,3350,644555
9,XGBoost,0.946612,+0.02,0.966612,0.974517,0.904322,0.938108,87,352,3327,644572




임계값 민감도 최종 요약


,algorithm,optimal_threshold,optimal_f1,minimum_f1_around_threshold,maximum_f1_drop,recall_range,fp_change_range,fn_change_range
0,LightGBM,0.971298,0.940278,0.923810,0.016468,0.063604,157,234
1,XGBoost,0.946612,0.941012,0.938108,0.002904,0.020386,74,75


###### 임계값 안정성 결과

두 가지 안정성 검사가 서로 다른 결과를 보임.

1. 시간에 따른 Fold별 임계값 안정성

선정 기준      	LightGBM 임계값 범위  	XGBoost 임계값 범위	    우세
Fold별 F1 최대	      0.018268	            0.061147       	LightGBM
Fold별 Recall≥0.93	  0.013938           	0.140540     	LightGBM

LightGBM의 Fold별 F1 최적 임계값은 서로 상당히 비슷. 반면 XGboost는 Fold 1 임계값이 크게 낮음. 
Recall 0.93 기준에서는 차이가 더 큼. 즉, 시간 구간이 달라졌을 때 동일한 임계값으로 작동하는 안정성은 LightGBM이 훨씬 좋음. XGBoost는 시간에 따라 예측확률의 위치가 많이 변한 것으로 보임.

2. 통합 임계값 주변 민감도

| 모델 | ±0.02 내 최대 F1 하락 | Recall 변동폭 | FP 변동폭 | FN 변동폭 |
|---|---:|---:|---:|---:|---:|
| LightGBM | 0.016468 | 0.063604 | 157 | 234 |
| XGBoost | 0.002904 | 0.020386 | 74 | 75 |

이 부분은 XGBoost가 우세합니다.

XGBoost는 통합 최적 임계값을 ±0.02 변경해도 F1 변화가 작음
LightGBM은 임계값이 높아질 때 Recall과 FN이 상대적으로 빠르게 변함

즉:

시간이 바뀌어도 비슷한 최적 임계값을 유지하는 모델: LightGBM
하나의 임계값 주변을 조금 조정했을 때 덜 민감한 모델: XGBoost


---


따라서 최종 알고리즘은 LightGBM으로 선정하는 것이 타당합니다.

특히 실제 FDS에서는 하나의 운영 임계값을 앞으로 들어오는 거래에 계속 적용해야 합니다. XGBoost처럼 Fold별 필요 임계값이 0.81~0.95까지 크게 바뀌는 것보다, LightGBM처럼 0.95~0.97 부근에서 유지되는 것이 운영상 유리합니다.
